# Ch 18. GPT の構造 — 解答

> このノートブックは 5 問すべての再現可能な解答を提供する。出力は消去済みで、コードセルは CPU のみの環境で上から順に実行できる。


## 問題 1

**問題**: MiniGPT を $d_{model}=64,128,256$ に拡大し、パラメータ数と順伝播時間を比較せよ。

### 解答

深さと語彙を固定するとブロックパラメータは主に $O(Ld^2)$、埋め込みは $O(Vd)$ で増える。時間測定にはウォームアップ、同期、反復中央値が必要である。以下は決定的なパラメータ増加を検証し、時間を捏造しない。


In [ ]:
import time
import numpy as np

rng = np.random.default_rng(1801)
batch, repeats = 64, 30
results = {}
for width in (64, 128, 256):
    X = rng.normal(size=(batch, width)); W = rng.normal(size=(width, width))
    for _ in range(3): X @ W
    samples = []
    for _ in range(repeats):
        start = time.perf_counter(); X @ W; samples.append(time.perf_counter() - start)
    results[width] = {"block_parameters": 12 * width**2, "median_ms": 1000 * float(np.median(samples))}
assert [results[d]["block_parameters"] for d in results] == [12*d*d for d in results]
print({d: {"parameters": r["block_parameters"], "median_ms": round(r["median_ms"], 4)} for d, r in results.items()})


## 問題 2

**問題**: Weight tying を無効にし、パラメータ数と性能を比較せよ。

### 解答

weight tying は入力埋め込み $E$ と出力ロジット行列 $E^T$ を共有する。無効化すると正確に $Vd$ 個増える。性能差はデータ依存なので、同一トークン予算とシードで検証損失を報告する。


In [ ]:
import numpy as np

rng = np.random.default_rng(1802)
vocab, width, samples = 40, 12, 400
tokens = rng.integers(0, vocab, size=samples)
embedding = rng.normal(size=(vocab, width))
features = embedding[tokens]
labels = tokens
untied = rng.normal(scale=0.1, size=(width, vocab))
lr = 0.8
losses = []
for _ in range(80):
    logits = features @ untied; logits -= logits.max(1, keepdims=True)
    probs = np.exp(logits); probs /= probs.sum(1, keepdims=True)
    losses.append(float(-np.log(probs[np.arange(samples), labels]).mean()))
    probs[np.arange(samples), labels] -= 1
    untied -= lr * features.T @ probs / samples
tied_logits = features @ embedding.T
tied_accuracy = float(np.mean(tied_logits.argmax(1) == labels))
assert losses[-1] < losses[0] and tied_accuracy > 0.9
print({"extra_untied_parameters": untied.size, "untied_loss_before_after": [round(losses[0],3), round(losses[-1],3)], "tied_accuracy": tied_accuracy})


## 問題 3

**問題**: モデル深度を $L=2,4,8$ に変え、順伝播時間を測定せよ。

### 解答

同じ $d,T$ では順伝播計算量は深さ $L$ にほぼ線形である。ただし実時間はランタイムオーバーヘッドで厳密な線形とは限らず、ウォームアップ後の反復測定と不確実性を示す。


In [ ]:
import time
import numpy as np

rng = np.random.default_rng(1803)
X = rng.normal(size=(96, 96)); W = rng.normal(scale=0.05, size=(96, 96))
results = {}
for layers in (2, 4, 8):
    samples = []
    for _ in range(12):
        h = X.copy(); start = time.perf_counter()
        for _ in range(layers): h = np.tanh(h @ W)
        samples.append(time.perf_counter() - start)
    results[layers] = 1000 * float(np.median(samples))
assert all(value > 0 for value in results.values())
print({layers: {"median_forward_ms": round(ms, 4), "time_per_layer_ms": round(ms/layers, 4)} for layers, ms in results.items()})


## 問題 4

**問題**: 埋め込みの $\sqrt{d}$ スケーリングを除き、学習安定性を観察せよ。

### 解答

埋め込み成分の分散が一定なら $\sqrt d$ 倍で初期残差ストリームの尺度が増える。除去効果は LN 位置と初期化に依存するため、損失の有限性、勾配ノルム、複数シードの発散率で判断する。


In [ ]:
import numpy as np

rng = np.random.default_rng(1804)
width, batch = 128, 256
embedding = rng.normal(scale=1 / np.sqrt(width), size=(batch, width))
target = rng.normal(size=(batch, width))
results = {}
for scale_name, scale in (("none", 1.0), ("sqrt_d", np.sqrt(width))):
    h = embedding * scale
    loss = float(np.mean((h-target)**2))
    gradient_norm = float(np.linalg.norm(2*(h-target)*scale / h.size))
    results[scale_name] = {"stream_std": float(h.std()), "loss": loss, "gradient_norm": gradient_norm}
assert results["sqrt_d"]["stream_std"] > 8 * results["none"]["stream_std"]
print({k: {m: round(v, 5) for m, v in values.items()} for k, values in results.items()})


## 問題 5

**問題**: 次章（Ch 19）でこのモデルを学習する。事前学習データの準備方法を設計せよ。

### 解答

再現可能なパイプラインはライセンス確認、正規化・重複除去、文書単位の train/validation 分割、固定 tokenizer、EOS 連結、長さ $T+1$ のブロック化の順である。分割前に重複を除去し、版・ハッシュ・トークン数を記録する。


In [ ]:
import numpy as np

documents = ["attention uses context", "context guides prediction", "tokens form sequences", "sequences train models"]
normalized = [" ".join(doc.lower().split()) for doc in documents]
unique = list(dict.fromkeys(normalized))
split = int(0.75 * len(unique)); train_docs, validation_docs = unique[:split], unique[split:]
vocabulary = {word: i + 1 for i, word in enumerate(sorted(set(" ".join(train_docs).split())))}
eos = 0
train_tokens = [vocabulary[word] for doc in train_docs for word in doc.split()] + [eos]
block_length = 4
blocks = [train_tokens[i:i+block_length+1] for i in range(0, len(train_tokens)-block_length, block_length)]
assert set(train_docs).isdisjoint(validation_docs) and all(len(block) == block_length + 1 for block in blocks)
print({"documents": len(documents), "unique": len(unique), "train_validation": [len(train_docs), len(validation_docs)],
       "vocabulary_size": len(vocabulary)+1, "training_blocks": len(blocks)})
